# INC-006 Incremental Mode and Compare Review

This notebook exercises the INC-006 contract (`specs/INC-006.md`): `run_incremental` (reprocess only changed raw files) and the read-only `compare` command. It works on temporary copies of the raw data and of the pipeline output, so neither `data/` nor an existing `output/` directory is modified. A full rebuild of the same raw copy is the reference for every incremental result.

Set `INC006_DATA_DIR` to override `data/`. The notebook runs four full rebuilds (about 80 s each) and several incremental runs, so allow roughly 6 to 8 minutes and about 2 GB of free disk space.

## Review Sections
1. Environment, helpers and raw-data fingerprint.
2. Scenario A: a baseline full run, then an incremental run with no change (no-op).
3. Scenario B: one corrected record; incremental versus full rebuild.
4. Scenario C: a new raw file; incremental versus full rebuild.
5. Scenario D: an extra day of records; incremental versus full rebuild.
6. The compare command detects differences (library and CLI exit codes).
7. A failing incremental run leaves the previous output unchanged.
8. Timings, raw-data immutability and acceptance summary with PASS/FAIL/NOT RUN.

Only table names, counts, participant ids and dates are displayed; raw record values are not printed.

In [1]:
import hashlib
import html
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import duckdb
from IPython.display import HTML, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'wearable_pipeline').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))
from wearable_pipeline.compare import compare_outputs, format_comparison
from wearable_pipeline.incremental import run_incremental
from wearable_pipeline.pipeline import ALL_GOLD_COLUMNS, PipelineError, run_pipeline

DATA_DIR = Path(os.environ.get('INC006_DATA_DIR', ROOT / 'data')).resolve()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Raw data directory not found: {DATA_DIR}')
WORK = Path(tempfile.mkdtemp(prefix='inc006-review-'))
acceptance = {}
timings = []
STEPS_COLUMNS = {
    'total_steps', 'steps_valid_minute_count', 'minutes_rest_to_moderate',
    'minutes_burst', 'steps_present', 'steps_quality_flags',
}


def render_table(title, columns, rows):
    header = ''.join('<th>' + html.escape(str(column)) + '</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join('<td>' + html.escape(str(value)) + '</td>' for value in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<h4>' + html.escape(title) + '</h4><table><thead><tr>' + header
        + '</tr></thead><tbody>' + body + '</tbody></table>'
    ))


def status(ok):
    return 'PASS' if ok else 'FAIL'


def tree_hash(path):
    digest = hashlib.sha256()
    for file in sorted(Path(path).rglob('*')):
        if file.is_file():
            digest.update(str(file.relative_to(path)).encode())
            digest.update(hashlib.sha256(file.read_bytes()).digest())
    return digest.hexdigest()


def timed(label, function, *args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    timings.append((label, round(time.perf_counter() - start, 1)))
    return result


def gold_changes(left_dir, right_dir):
    # Gold keys whose row differs between two outputs, and the gold columns that differ.
    con = duckdb.connect(':memory:')
    con.execute('ATTACH $$' + str(Path(left_dir) / 'wearable.duckdb') + '$$ AS lhs (READ_ONLY)')
    con.execute('ATTACH $$' + str(Path(right_dir) / 'wearable.duckdb') + '$$ AS rhs (READ_ONLY)')
    try:
        keys = con.execute(
            '''SELECT DISTINCT participant_id, local_date FROM (
                   (SELECT * FROM lhs.gold_clinical_daily EXCEPT ALL SELECT * FROM rhs.gold_clinical_daily)
                   UNION ALL
                   (SELECT * FROM rhs.gold_clinical_daily EXCEPT ALL SELECT * FROM lhs.gold_clinical_daily)
               ) ORDER BY 1, 2'''
        ).fetchall()
        columns = []
        for name, _ in ALL_GOLD_COLUMNS[2:]:
            count = con.execute(
                'SELECT COUNT(*) FROM lhs.gold_clinical_daily l JOIN rhs.gold_clinical_daily r '
                'USING (participant_id, local_date) WHERE l.' + name + ' IS DISTINCT FROM r.' + name
            ).fetchone()[0]
            if count:
                columns.append(name)
        return keys, columns
    finally:
        con.close()


def gold_row_count(directory):
    con = duckdb.connect(str(Path(directory) / 'wearable.duckdb'), read_only=True)
    try:
        return con.execute('SELECT COUNT(*) FROM gold_clinical_daily').fetchone()[0]
    finally:
        con.close()


RAW_HASH_BEFORE = tree_hash(DATA_DIR)
print(f'Raw data directory: {DATA_DIR}')
print(f'Raw data fingerprint before: {RAW_HASH_BEFORE[:16]}')
print(f'Temporary work directory: {WORK}')
print(f'DuckDB version: {duckdb.__version__}')

Raw data directory: /home/michaelmadeira/wearable-health-data-pipeline/data
Raw data fingerprint before: abd71e9575b8ae2a
Temporary work directory: /tmp/inc006-review-pbdnxz8d
DuckDB version: 1.5.6


In [2]:
# Scenario A: baseline full run, then an incremental run with nothing changed.
data_a = WORK / 'data_a'
shutil.copytree(DATA_DIR, data_a)
out_a = WORK / 'out_a'
full_a = timed('A: full rebuild (baseline)', run_pipeline, data_a, out_a)
snapshot_a = WORK / 'out_a_snapshot'
shutil.copytree(out_a, snapshot_a)
quality_before = tree_hash(out_a / 'quality')

noop = timed('A: incremental, nothing changed', run_incremental, data_a, out_a)
noop_compare = compare_outputs(snapshot_a, out_a)
noop_checks = [
    ('Outcome is noop', noop.outcome == 'noop', noop.outcome),
    ('No gold row recomputed', noop.affected_gold_rows == 0, noop.affected_gold_rows),
    ('Run id unchanged', noop.run_id == full_a.run_id, noop.run_id),
    ('Every table identical to the pre-run snapshot', noop_compare.identical, format_comparison(noop_compare).strip()),
    ('Quality directory bytes unchanged', tree_hash(out_a / 'quality') == quality_before, ''),
]
render_table(
    'Scenario A: no-op incremental run',
    ['check', 'status', 'detail'],
    [(name, status(ok), detail) for name, ok, detail in noop_checks],
)
acceptance['No-op incremental run changes no table, report byte or run id'] = all(ok for _, ok, _ in noop_checks)

check,status,detail
Outcome is noop,PASS,noop
No gold row recomputed,PASS,0
Run id unchanged,PASS,a826c8397cf5ad8e
Every table identical to the pre-run snapshot,PASS,identical
Quality directory bytes unchanged,PASS,


In [3]:
# Scenario B: one corrected record in P001's steps file (timestamp 2026-01-20T12:00:00, +37 steps).
data_b = WORK / 'data_b'
shutil.copytree(data_a, data_b)
steps_path = data_b / 'wearable_events' / 'P001' / 'steps.json'
records = json.loads(steps_path.read_text())
target = next(i for i, record in enumerate(records) if record['timestamp'] == '2026-01-20T12:00:00')
records[target]['steps'] += 37
steps_path.write_text(json.dumps(records))

out_b = WORK / 'out_b'
shutil.copytree(snapshot_a, out_b)
result_b = timed('B: incremental, one record corrected', run_incremental, data_b, out_b)
out_b_full = WORK / 'out_b_full'
full_b = timed('B: full rebuild of the same copy', run_pipeline, data_b, out_b_full)
compare_b = compare_outputs(out_b, out_b_full)
keys_b, columns_b = gold_changes(snapshot_a, out_b)
checks_b = [
    ('Outcome is incremental', result_b.outcome == 'incremental', result_b.outcome),
    ('Only the changed file is parsed', result_b.changed_paths == ('wearable_events/P001/steps.json',), result_b.changed_paths),
    ('Incremental output identical to a full rebuild', compare_b.identical, format_comparison(compare_b).strip()),
    ('Run id equals the full rebuild run id', result_b.run_id == full_b.run_id, result_b.run_id),
    ('Run id differs from the baseline', result_b.run_id != full_a.run_id, ''),
    ('Exactly one gold row changed against the baseline (P001, 2026-01-20)', [(p, str(d)) for p, d in keys_b] == [('P001', '2026-01-20')], keys_b),
    ('Only steps columns changed', set(columns_b) <= STEPS_COLUMNS, columns_b),
]
render_table(
    'Scenario B: one corrected record',
    ['check', 'status', 'detail'],
    [(name, status(ok), detail) for name, ok, detail in checks_b],
)
acceptance['Corrected record: incremental equals full rebuild'] = compare_b.identical and result_b.run_id == full_b.run_id
acceptance['Corrected record: only that participant-day gold row changes'] = [(p, str(d)) for p, d in keys_b] == [('P001', '2026-01-20')] and set(columns_b) <= STEPS_COLUMNS

check,status,detail
Outcome is incremental,PASS,incremental
Only the changed file is parsed,PASS,"('wearable_events/P001/steps.json',)"
Incremental output identical to a full rebuild,PASS,identical
Run id equals the full rebuild run id,PASS,41c339cef2ec4025
Run id differs from the baseline,PASS,
"Exactly one gold row changed against the baseline (P001, 2026-01-20)",PASS,"[('P001', datetime.date(2026, 1, 20))]"
Only steps columns changed,PASS,"['total_steps', 'minutes_rest_to_moderate', 'minutes_burst']"


In [4]:
# Scenario C: a new raw file. The baseline is a full run on a copy without P050's sleep file; the file is then added.
data_c = WORK / 'data_c'
shutil.copytree(data_a, data_c)
sleep_path = data_c / 'wearable_events' / 'P050' / 'sleep.json'
sleep_path.unlink()
out_c = WORK / 'out_c'
timed('C: full rebuild (file missing)', run_pipeline, data_c, out_c)
snapshot_c = WORK / 'out_c_snapshot'
shutil.copytree(out_c, snapshot_c)
shutil.copy2(data_a / 'wearable_events' / 'P050' / 'sleep.json', sleep_path)

result_c = timed('C: incremental, file added', run_incremental, data_c, out_c)
compare_c = compare_outputs(out_c, out_a)  # data_c now equals data_a, whose full run is out_a
keys_c, columns_c = gold_changes(snapshot_c, out_c)
checks_c = [
    ('Outcome is incremental', result_c.outcome == 'incremental', result_c.outcome),
    ('The new file is the only parsed path', result_c.new_paths == ('wearable_events/P050/sleep.json',) and not result_c.changed_paths, result_c.new_paths),
    ('Incremental output identical to a full rebuild', compare_c.identical, format_comparison(compare_c).strip()),
    ('Run id equals the baseline full run id', result_c.run_id == full_a.run_id, result_c.run_id),
    ('Only P050 gold rows changed', {p for p, _ in keys_c} == {'P050'}, len(keys_c)),
    ('Only sleep columns changed', all(name.startswith(('total_sleep', 'deep_sleep', 'sleep_')) for name in columns_c), columns_c),
]
render_table(
    'Scenario C: new raw file',
    ['check', 'status', 'detail'],
    [(name, status(ok), detail) for name, ok, detail in checks_c],
)
acceptance['New file: incremental equals full rebuild'] = compare_c.identical and result_c.run_id == full_a.run_id
acceptance['New file: only that participant\'s gold rows change'] = {p for p, _ in keys_c} == {'P050'}

check,status,detail
Outcome is incremental,PASS,incremental
The new file is the only parsed path,PASS,"('wearable_events/P050/sleep.json',)"
Incremental output identical to a full rebuild,PASS,identical
Run id equals the baseline full run id,PASS,a826c8397cf5ad8e
Only P050 gold rows changed,PASS,28
Only sleep columns changed,PASS,"['total_sleep_min', 'deep_sleep_pct', 'sleep_efficiency_pct', 'sleep_session_count', 'sleep_present', 'sleep_quality_flags']"


In [5]:
# Scenario D: an extra day (2026-02-05, outside the 28-day window) appended to P002's steps file.
data_d = WORK / 'data_d'
shutil.copytree(data_a, data_d)
extra_path = data_d / 'wearable_events' / 'P002' / 'steps.json'
records = json.loads(extra_path.read_text())
records += [
    {'timestamp': f'2026-02-05T{hour:02d}:{minute:02d}:00', 'steps': 5, 'participant_id': 'P002'}
    for hour in range(24) for minute in range(0, 60, 15)
]
extra_path.write_text(json.dumps(records))

out_d = WORK / 'out_d'
shutil.copytree(snapshot_a, out_d)
result_d = timed('D: incremental, extra day', run_incremental, data_d, out_d)
out_d_full = WORK / 'out_d_full'
full_d = timed('D: full rebuild of the same copy', run_pipeline, data_d, out_d_full)
compare_d = compare_outputs(out_d, out_d_full)
keys_d, _ = gold_changes(snapshot_a, out_d)
checks_d = [
    ('Outcome is incremental', result_d.outcome == 'incremental', result_d.outcome),
    ('Incremental output identical to a full rebuild', compare_d.identical, format_comparison(compare_d).strip()),
    ('Run id equals the full rebuild run id', result_d.run_id == full_d.run_id, result_d.run_id),
    ('Gold row count unchanged (the extra day is outside the window)', gold_row_count(out_d) == gold_row_count(snapshot_a), gold_row_count(out_d)),
    ('No gold row changed', not keys_d, keys_d),
]
render_table(
    'Scenario D: extra day of records',
    ['check', 'status', 'detail'],
    [(name, status(ok), detail) for name, ok, detail in checks_d],
)
acceptance['Extra day: incremental equals full rebuild'] = compare_d.identical and result_d.run_id == full_d.run_id

check,status,detail
Outcome is incremental,PASS,incremental
Incremental output identical to a full rebuild,PASS,identical
Run id equals the full rebuild run id,PASS,b7beb2ce9b14f2e9
Gold row count unchanged (the extra day is outside the window),PASS,1400
No gold row changed,PASS,[]


In [6]:
# Section 6: the compare command detects differences and exits with the documented codes.
differing = compare_outputs(snapshot_a, out_b)
print(format_comparison(differing))
env = {**os.environ, 'PYTHONPATH': str(ROOT / 'src')}


def cli_compare(left, right):
    return subprocess.run(
        [sys.executable, '-m', 'wearable_pipeline', 'compare', str(left), str(right)],
        capture_output=True, text=True, cwd=ROOT, env=env,
    )


same_cli = cli_compare(out_b, out_b_full)
different_cli = cli_compare(snapshot_a, out_b)
missing_cli = cli_compare(snapshot_a, WORK / 'does-not-exist')
tables_differing = [diff.table for diff in differing.table_diffs]
checks_cmp = [
    ('Identical outputs: exit 0 and the text identical', same_cli.returncode == 0 and same_cli.stdout.strip() == 'identical', same_cli.returncode),
    ('Different outputs: exit 1 and the first line differ', different_cli.returncode == 1 and different_cli.stdout.splitlines()[0] == 'differ', different_cli.returncode),
    ('Unreadable output: exit 1 and an error on stderr', missing_cli.returncode == 1 and missing_cli.stderr.startswith('error:'), missing_cli.returncode),
    ('The differing tables are named (report and tables only)', len(tables_differing) > 0 and 'gold_clinical_daily' in tables_differing, tables_differing),
    ('Differing outputs also differ in the quality report bytes', len(differing.report_diffs) > 0, differing.report_diffs),
]
render_table(
    'Compare command',
    ['check', 'status', 'detail'],
    [(name, status(ok), detail) for name, ok, detail in checks_cmp],
)
acceptance['Compare reports identical (exit 0), different and unreadable (exit 1) outputs'] = all(ok for _, ok, _ in checks_cmp)

differ
table gold_clinical_daily: rows differ: left_rows=1400 right_rows=1400 left_only=1 right_only=1
table quality_rejects: rows differ: left_rows=26368 right_rows=26368 left_only=26368 right_only=26368
table quality_results: rows differ: left_rows=41 right_rows=41 left_only=41 right_only=41
table raw_manifest: rows differ: left_rows=154 right_rows=154 left_only=1 right_only=1
table silver_steps_minute: rows differ: left_rows=2014950 right_rows=2014950 left_only=1 right_only=1
report quality/report.md: differs



check,status,detail
Identical outputs: exit 0 and the text identical,PASS,0
Different outputs: exit 1 and the first line differ,PASS,1
Unreadable output: exit 1 and an error on stderr,PASS,1
The differing tables are named (report and tables only),PASS,"['gold_clinical_daily', 'quality_rejects', 'quality_results', 'raw_manifest', 'silver_steps_minute']"
Differing outputs also differ in the quality report bytes,PASS,"(('quality/report.md', 'differs'),)"


In [7]:
# Section 7: a failing incremental run leaves the previous output byte-identical.
# P002's heart-rate file in the scenario B copy is made invalid, so the incremental parse must fail.
(data_b / 'wearable_events' / 'P002' / 'heart_rate.json').write_text('{ not valid json')
state_before = tree_hash(out_b)
try:
    run_incremental(data_b, out_b)
    raised = False
except PipelineError:
    raised = True
state_after = tree_hash(out_b)
render_table(
    'Failure atomicity',
    ['check', 'status'],
    [
        ('Invalid raw file raises PipelineError', status(raised)),
        ('Database and report byte-identical after the failure', status(state_before == state_after)),
    ],
)
acceptance['A failing incremental run raises and leaves the previous output unchanged'] = raised and state_before == state_after

check,status
Invalid raw file raises PipelineError,PASS
Database and report byte-identical after the failure,PASS


In [8]:
# Section 8: timings, raw-data immutability and acceptance summary.
render_table('Timings (seconds)', ['step', 'seconds'], timings)
full_times = [seconds for label, seconds in timings if 'full rebuild' in label]
incremental_times = [seconds for label, seconds in timings if 'incremental' in label]
acceptance['Incremental runs are faster than the full rebuilds'] = max(incremental_times) < min(full_times)

raw_hash_after = tree_hash(DATA_DIR)
print(f'Raw data fingerprint after: {raw_hash_after[:16]}')
acceptance['Raw data directory unchanged'] = raw_hash_after == RAW_HASH_BEFORE

rows = [
    (name, 'PASS' if result is True else 'FAIL' if result is False else 'NOT RUN')
    for name, result in acceptance.items()
]
rows.append(('Failure after the commit leaves a stale report (known low finding)', 'NOT RUN here; see the implementation review'))
rows.append(('Old-schema and window-change fallbacks, cross-file ownership', 'NOT RUN here; covered by the pipeline tests'))
render_table('INC-006 review summary', ['acceptance check', 'status'], rows)
shutil.rmtree(WORK, ignore_errors=True)
print('Review is observational; only temporary copies were run and they have been removed.')

step,seconds
A: full rebuild (baseline),55.4
"A: incremental, nothing changed",0.3
"B: incremental, one record corrected",2.0
B: full rebuild of the same copy,54.7
C: full rebuild (file missing),53.1
"C: incremental, file added",1.9
"D: incremental, extra day",2.0
D: full rebuild of the same copy,55.0


Raw data fingerprint after: abd71e9575b8ae2a


acceptance check,status
"No-op incremental run changes no table, report byte or run id",PASS
Corrected record: incremental equals full rebuild,PASS
Corrected record: only that participant-day gold row changes,PASS
New file: incremental equals full rebuild,PASS
New file: only that participant's gold rows change,PASS
Extra day: incremental equals full rebuild,PASS
"Compare reports identical (exit 0), different and unreadable (exit 1) outputs",PASS
A failing incremental run raises and leaves the previous output unchanged,PASS
Incremental runs are faster than the full rebuilds,PASS
Raw data directory unchanged,PASS


Review is observational; only temporary copies were run and they have been removed.
